In [1]:
from script.dataset import MaskedLMDataSet
from script.model.encoder import MaskedLM

import pandas as pd 
import numpy as np
import os
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from script.dataprocess import mk_aa_dict
from script.utils import EarlyStopping

In [2]:
tcr = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC_TCR/unlabel_TCRb_test.csv',header=0,index_col=0, nrows=10000)
peptide = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC/unique_peptide_test.csv',header=0,index_col=0, nrows=10000)

In [3]:
aa_dict = mk_aa_dict()
max_len = 20
d_seq = 128
d_pair = 64
d_head = 32
dropout = 0.1
n_layers = 6

In [4]:
tcr_lm = MaskedLM(aa_size=len(aa_dict), max_len=max_len, 
                    d_seq=d_seq, d_head_seq=d_head, 
                    d_pair=d_pair, d_head_pair=d_head, 
                    dropout=dropout, n_layers=n_layers)

In [5]:
# total_params = sum(p.numel() for p in tcr_lm.parameters())
# trainable_params = sum(p.numel() for p in tcr_lm.parameters() if p.requires_grad)

# print("total_params (m):", total_params/1e6)
# print("trainable_params (m):", trainable_params/1e6)


In [6]:
masked_rate=0.15
batch_size=32   
lr=5e-5
weight_decay=0.01
max_epoch=500
device='mps'
patience=50
outdir='tcr_test.pt'

In [7]:
tcr_dataset = MaskedLMDataSet(seq_list=tcr['cdr3'].tolist(), 
            
                              max_len=max_len,
                                masked_rate=0.15, contiguous_prob=0.1)

In [8]:
tcr_dataloader = DataLoader(tcr_dataset, batch_size=batch_size, 
                              shuffle=False, num_workers=4)

In [9]:
tcr_lm.to(device)

optimizer = torch.optim.AdamW(tcr_lm.parameters(), lr=lr, weight_decay=weight_decay)
loss_dict = {}
t = tqdm(range(max_epoch), desc="Epochs")
for epoch in t: 
 
    tcr_lm.train()
    epoch_loss = 0.0
    for idx, (x, _) in enumerate(tcr_dataloader):
        input = x['mlm_input'].to(device)
        label = x['mlm_label'].to(device)
 
        _, mlm_loss = tcr_lm(input, label)
        
        optimizer.zero_grad()
        mlm_loss.backward()
        optimizer.step()
        
        epoch_loss += mlm_loss.item()

    print(round(epoch_loss/(idx+1),3))
            

Epochs:   0%|          | 1/500 [01:44<14:29:45, 104.58s/it]

4.785


Epochs:   0%|          | 2/500 [03:30<14:32:29, 105.12s/it]

2.184


Epochs:   1%|          | 3/500 [05:15<14:30:09, 105.05s/it]

...

1.416


Epochs:   3%|▎         | 16/500 [28:15<14:18:29, 106.42s/it]

1.402


Epochs:   3%|▎         | 17/500 [30:01<14:16:15, 106.37s/it]

1.397
